In [4]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import HeteroData
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [5]:
acc = pd.read_csv(r'C:\Users\garv\OneDrive\Desktop\Work Folder\Fraud Detection\Datasets\HI-Small_accounts.csv')
data = pd.read_csv(r'C:\Users\garv\OneDrive\Desktop\Work Folder\Fraud Detection\Datasets\HI-Small_Trans.csv')

In [6]:
data.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [7]:
acc.tail()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
518576,France Bank #33,3881,807886B70,80062D160,Sole Proprietorship #22257
518577,National Bank of Topeka,333423,81314C870,800F40230,Sole Proprietorship #4995
518578,Plandor Trust Bank,1467,804ED2270,800CAC4C0,Sole Proprietorship #35326
518579,Sappo Bancorp,2843,801727270,800EFCB40,Sole Proprietorship #3522
518580,Savings Bank of Montpelier,213580,804F6D9F0,800406D60,Partnership #49800


## Accounts Cleaning

In [8]:
acc['Entity Name'] = acc['Entity Name'].apply(lambda x: x.split('#')[0].strip())

In [9]:
acc['Entity Name'].value_counts()

Entity Name
Partnership            189683
Corporation            172351
Sole Proprietorship    149048
Country                  6692
Individual                740
Direct                     67
Name: count, dtype: int64

In [10]:
acc.drop(['Bank Name', 'Entity ID'], axis=1, inplace=True)

In [11]:
acc.sample(5)

,Bank ID,Account Number,Entity Name
248257,43818,8107E44E0,Partnership
366119,33361,809C48DA0,Corporation
90557,33323,80D607170,Corporation
89082,241121,80F6FAFB0,Corporation
31226,18617,8091C27D0,Sole Proprietorship


## Transactions Cleaning

In [12]:
data

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,US Dollar,3697.340000,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,US Dollar,0.010000,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,US Dollar,14675.570000,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,US Dollar,2806.970000,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,US Dollar,36682.970000,US Dollar,Reinvestment,0
...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,Bitcoin,0.154978,Bitcoin,Bitcoin,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,Bitcoin,0.108128,Bitcoin,Bitcoin,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,Bitcoin,0.004988,Bitcoin,Bitcoin,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,Bitcoin,0.038417,Bitcoin,Bitcoin,0


In [13]:
data['Timestamp'] = data['Timestamp'].apply(lambda x: (int(x.split()[0][-2:]) - 1) * 1440 + int(x.split()[1].split(':')[0]) * 60 + int(x.split()[1].split(':')[1]))

In [14]:
data['day'] = (data['Timestamp'] // 1440) + 1
data['hour'] = (data['Timestamp'] % 1440) // 60

In [15]:
data.sort_values(by='Timestamp', inplace=True)

In [16]:
data['sender_txn_daily_count'] = data.groupby(['Account', 'day']).cumcount() + 1
data['receiver_txn_daily_count'] = data.groupby(['Account.1', 'day']).cumcount() + 1

data['sender_txn_daily_amount'] = data.groupby(['Account', 'day'])['Amount Paid'].cumsum()
data['receiver_txn_daily_amount'] = data.groupby(['Account.1', 'day'])['Amount Received'].cumsum()

In [ ]:
data.drop(['Timestamp'], axis=1, inplace=True)
data.rename({'Account': 'sender', 'Account.1': 'receiver'}, axis=1, ignore_index=True, inplace=True)

In [18]:
data.sample(5)

,From Bank,sender,To Bank,receiver,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,day,hour,sender_txn_daily_count,receiver_txn_daily_count,sender_txn_daily_amount,receiver_txn_daily_amount
4872719,219184,807547B60,146713,81327EC20,46.31,US Dollar,46.31,US Dollar,Credit Card,0,10,0,1,2,46.31,47.89
2298433,25309,80DDFA4B0,123308,80BF44FE0,62.51,Euro,62.51,Euro,Credit Card,0,5,0,1,1,62.51,62.51
148475,25536,805586510,25536,805586510,11.56,Euro,11.56,Euro,Reinvestment,0,1,0,1,1,11.56,11.56
4120908,12381,805583AD0,12585,80EB5DA60,299763.52,US Dollar,299763.52,US Dollar,ACH,0,8,19,7,4,811265.94,778561.96
1546941,10,8001E37F0,701,80054ABE0,1358.11,US Dollar,1358.11,US Dollar,Credit Card,0,2,13,14,2,30510.99,2249.77


## Adding daily metrices

In [19]:
# Compute daily aggregates per sender
sender_daily = data.groupby(['sender', 'day', 'From Bank']).agg(
    sender_daily_count=('sender_txn_daily_count', 'max'),
    sender_daily_amount=('sender_txn_daily_amount', 'max')
).reset_index()

sender_avg = sender_daily.groupby(['sender', 'From Bank']).agg(
    avg_daily_txn_sent=('sender_daily_count', 'mean'),
    avg_daily_amount_sent=('sender_daily_amount', 'mean')
).reset_index().rename(columns={'sender': 'Account Number', 'From Bank': 'Bank ID'})

# Compute daily aggregates per receiver
receiver_daily = data.groupby(['receiver', 'day', 'To Bank']).agg(
    receiver_daily_count=('receiver_txn_daily_count', 'max'),
    receiver_daily_amount=('receiver_txn_daily_amount', 'max')
).reset_index()

receiver_avg = receiver_daily.groupby(['receiver', 'To Bank']).agg(
    avg_daily_txn_received=('receiver_daily_count', 'mean'),
    avg_daily_amount_received=('receiver_daily_amount', 'mean')
).reset_index().rename(columns={'receiver': 'Account Number', 'To Bank': 'Bank ID'})

In [20]:
acc = acc.merge(sender_avg, on=['Account Number', 'Bank ID'], how='left')
acc = acc.merge(receiver_avg, on=['Account Number', 'Bank ID'], how='left')

acc[['avg_daily_txn_sent', 'avg_daily_amount_sent']] = acc[['avg_daily_txn_sent', 'avg_daily_amount_sent']].fillna(0)
acc[['avg_daily_txn_received', 'avg_daily_amount_received']] = acc[['avg_daily_txn_received', 'avg_daily_amount_received']].fillna(0)

In [21]:
acc.sample(3)

,Bank ID,Account Number,Entity Name,avg_daily_txn_sent,avg_daily_amount_sent,avg_daily_txn_received,avg_daily_amount_received
294510,323278,8087BEE80,Corporation,1.0,21320.16,1.0,21320.160
49924,148785,812825D90,Corporation,1.0,13203.10,3.5,7554.317
31994,14011,8116323E0,Corporation,0.0,0.00,1.0,22.760


In [31]:
data.sample(3)

,From Bank,sender,To Bank,receiver,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,day,hour,sender_txn_daily_count,receiver_txn_daily_count,sender_txn_daily_amount,receiver_txn_daily_amount
3340821,14,80300FDC0,1267,80393E080,23828.83,Yuan,23828.83,Yuan,Credit Card,0,7,3,3,3,6.616228e+05,661622.75
3673373,70,1004289C0,243614,80FFD5010,8992.27,Shekel,8992.27,Shekel,Cheque,0,7,20,1657,3,3.353081e+08,48770.44
3410172,1,80387F580,223233,808938340,4384.37,US Dollar,4384.37,US Dollar,Cheque,0,7,7,7,1,9.240050e+03,4384.37


In [36]:
acc['Is Laundering'] = 0

In [38]:
for d in np.where(data['Is Laundering'] == 1)[0]:
    sender, receiver, from_bank, to_bank = data.iloc[d][['sender', 'receiver', 'From Bank', 'To Bank']]
    acc.loc[(acc['Account Number'] == sender) & (acc['Bank ID'] == from_bank), 'Is Laundering'] = 1
    acc.loc[(acc['Account Number'] == receiver) & (acc['Bank ID'] == to_bank), 'Is Laundering'] = 1

In [39]:
acc['Is Laundering'].value_counts()

Is Laundering
0    512224
1      6357
Name: count, dtype: int64

In [46]:
acc.sample(3)

,Bank ID,Account Number,Entity Name,avg_daily_txn_sent,avg_daily_amount_sent,avg_daily_txn_received,avg_daily_amount_received,Is Laundering
133836,217096,8073A1430,Corporation,1.0,1.008301e+08,1.0,1.008301e+08,0
434482,27616,80D9B43F0,Corporation,1.9,1.840102e+03,1.0,6.596176e+04,0
332742,344794,8115C4D50,Corporation,1.0,9.965400e+02,0.0,0.000000e+00,0


In [44]:
data.sample(3)

,From Bank,sender,To Bank,receiver,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,day,hour,sender_txn_daily_count,receiver_txn_daily_count,sender_txn_daily_amount,receiver_txn_daily_amount
1276803,11318,809F926D0,211050,80C2142E0,120.88,US Dollar,120.88,US Dollar,Credit Card,0,2,4,1,3,120.88,14796.89
3167262,211,80A223DE0,27621,80A40E550,1343.38,Australian Dollar,1343.38,Australian Dollar,Cash,0,6,19,3,3,9711.24,9711.24
4468871,210,8090EE7D0,24856,809291D50,439532.38,Canadian Dollar,439532.38,Canadian Dollar,Cheque,0,9,8,4,3,611486.86,604646.87


## Creating Lables and Edge Index

In [48]:
labels = pd.DataFrame(acc['Is Laundering'].values, columns=['Is Laundering'])

In [ ]:
edge_index = data[['sender', 'receiver']]

## Creating nodes and edges features

In [55]:
acc.sample(3)

,Bank ID,Account Number,Entity Name,avg_daily_txn_sent,avg_daily_amount_sent,avg_daily_txn_received,avg_daily_amount_received,Is Laundering
424481,210261,8145C26D0,Partnership,1.000000,5070503.48,1.333333,2131364.090,0
148363,12797,80132B640,Corporation,5.857143,2215285.06,3.600000,1984.826,0
385822,34383,8104ED560,Partnership,1.000000,165.64,0.000000,0.000,0


In [56]:
data.sample(3)

,From Bank,sender,To Bank,receiver,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,day,hour,sender_txn_daily_count,receiver_txn_daily_count,sender_txn_daily_amount,receiver_txn_daily_amount
1455308,1,800791830,54717,8141C5B90,607.82,US Dollar,607.82,US Dollar,Cheque,0,2,10,12,5,2590012.53,2062.48
522210,24850,802A96EF0,24850,802A96EF0,11.60,Euro,11.60,Euro,Reinvestment,0,1,6,2,2,13208.72,13208.72
4441699,5425,802D99650,112212,80807D690,111.96,Euro,111.96,Euro,Cheque,0,9,7,2,2,318.99,318.99


In [58]:
nodes = acc.drop(['Entity Name', 'Account Number'], axis=1)

In [63]:
edges = data.drop(['From Bank', 'To Bank', 'sender', 'receiver', 'Payment Format', 'sender_txn_daily_count', 'receiver_txn_daily_count', 'sender_txn_daily_amount', 'receiver_txn_daily_amount'], axis=1)

In [64]:
nodes.to_csv('nodes.csv', index=False)
edges.to_csv('edges.csv', index=False)

In [61]:
labels.to_csv('labels.csv', index=False)
edge_index.to_csv('edge_index.csv', index=False)

In [62]:
data.to_csv('small-refined-data.csv', index=False)
acc.to_csv('small-refined-accounts.csv', index=False)

In [16]:
pd.concat([data['Receiving Currency'], data['Payment Currency']]).value_counts()

US Dollar            3774513
Euro                 2340314
Swiss Franc           472744
Yuan                  420303
Shekel                387172
Rupee                 382267
UK Pound              361993
Ruble                 312539
Yen                   311528
Bitcoin               294217
Canadian Dollar       281399
Australian Dollar     275280
Mexican Peso          221189
Saudi Riyal           178985
Brazil Real           142247
Name: count, dtype: int64

In [17]:
data.sample(5)

,From Bank,sender,To Bank,receiver,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,day,hour,sender_txn_daily_count,receiver_txn_daily_count,sender_txn_daily_amount,receiver_txn_daily_amount
3419938,2947,801BE6D80,112,80E9C5870,11.09,Swiss Franc,11.09,Swiss Franc,Wire,0,7,7,7,4,4.564350e+03,2186.67
4717807,338032,80DEAFD30,5328,80DE84F30,16.32,Euro,16.32,Euro,ACH,0,9,18,1,1,1.632000e+01,16.32
3669141,1502,8020E45D0,26818,8078A3BF0,19.97,Euro,19.97,Euro,Wire,0,7,20,3,2,1.067690e+03,27.21
4988690,70,1004286A8,127179,80BA74A10,1095.25,Euro,1095.25,Euro,Cheque,0,10,12,2445,1,3.287780e+08,1095.25
2397959,1,8005F3D00,21258,800833C50,60200.09,US Dollar,60200.09,US Dollar,Cheque,0,5,5,1,1,6.020009e+04,60200.09


In [18]:
acc.sample(5)

,Bank ID,Account Number,Entity Name
408197,5,8083649F0,Sole Proprietorship
476689,354214,814648811,Country
308006,137218,80E58BE60,Corporation
155826,34462,803512AA0,Partnership
28309,136012,80E89D610,Partnership


In [19]:
data[(data['Amount Received'] != data['Amount Paid'])]

,From Bank,sender,To Bank,receiver,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,day,hour,sender_txn_daily_count,receiver_txn_daily_count,sender_txn_daily_amount,receiver_txn_daily_amount
316477,15723,80F2E0350,15723,80F2E0350,396655.730000,Saudi Riyal,105743.84,US Dollar,ACH,0,1,0,2,1,502399.570000,396655.730000
97530,212048,8046B84F0,212048,8046B84F0,19.920000,US Dollar,2099.05,Yen,ACH,0,1,0,2,1,2118.970000,19.920000
207958,14325,806A3B500,14325,806A3B500,21271.070000,Yuan,3175.92,US Dollar,ACH,0,1,0,2,1,24446.990000,21271.070000
207956,14325,806A3B500,14325,806A3B500,32249.380000,Yuan,4815.07,US Dollar,ACH,0,1,0,4,2,61511.440000,53520.450000
320524,21611,80DA59911,21611,80DA59910,0.000001,Bitcoin,0.02,US Dollar,ACH,0,1,0,2,1,0.020001,0.000001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5033968,29404,8041A3440,29404,8041A3440,8053.240000,Euro,9436.65,US Dollar,ACH,0,17,14,3,2,30965.270000,18339.760000
4962227,9371,8043A0FB0,9371,8043A0FB0,2078.060000,Euro,2435.04,US Dollar,ACH,0,17,18,2,1,9064.530000,2078.060000
5033971,29404,8041A3440,29404,8041A3440,12376.110000,Euro,14502.12,US Dollar,ACH,0,18,6,2,1,21454.120000,12376.110000
5033973,29404,8041A3440,29404,8041A3440,2391.920000,Saudi Riyal,637.66,US Dollar,ACH,0,18,9,4,2,34467.890000,14768.030000


In [20]:
acc['Entity Name'].value_counts()

Entity Name
Partnership            189683
Corporation            172351
Sole Proprietorship    149048
Country                  6692
Individual                740
Direct                     67
Name: count, dtype: int64

In [47]:
data.groupby('Payment Format')['Is Laundering'].sum()

Payment Format
ACH             4483
Bitcoin           56
Cash             108
Cheque           324
Credit Card      206
Reinvestment       0
Wire               0
Name: Is Laundering, dtype: int64